# 06 — Fit and review the final timing models

**You will:** run PLEB again on the retained selection, inspect fresh diagnostics,
explicitly save fitted PAR files, and record a model-review checkpoint.
Running diagnostics does not by itself update the PAR files committed to a dataset.
The final model must be paired with the exact final TIM selection.

In [ ]:
from pathlib import Path
import os, sys, json, shutil, subprocess, hashlib, tarfile
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import tomlkit
COURSE = Path(os.environ['PLEB_COURSE_HOME'])
sys.path.insert(0, str(COURSE))
import release_tools as lab
ROOT = lab.environment(Path(os.environ['PLEB_LEARNING_OUTPUT']))
REPO = ROOT/'dataset-repo'
CONFIGS = ROOT/'configs'
PULSARS = lab.PULSARS
CLI = [sys.executable, '-m', 'pleb.cli']

In [ ]:
common = json.loads((CONFIGS/'common.json').read_text())
final_fit = dict(common,branches=['reviewed'],reference_branch='reviewed',
                 outdir_name='06_final_fit',run_tempo2=True,run_pqc=True)
detector = tomlkit.parse((CONFIGS/'03_detect.toml').read_text())
final_fit.update({k:v for k,v in detector.items() if k.startswith('pqc_')})
path = lab.write_config(CONFIGS/'06_final_fit.toml',final_fit)
lab.command(CLI+['--config',path],log=ROOT/'logs/06_final_fit.log')
print('Final diagnostics:',ROOT/'results/06_final_fit')

## Save fitted coefficients explicitly

The next command calls Tempo2 on the final copied input and requests `-newpar`.
Review its log, model coefficients, uncertainties and plots before committing that model.
It uses the same image, ephemeris, clock setup, epoch and selected observations.
Compare RMS/WRMS together with residual structure and the active-count audit. A low RMS
alone does not establish a useful release, and excluding valid data can lower it.
Repeated fits must stabilise on the same TOAs: we require at least three passes,
relative WRMS change below 1e-5 and ordinary fitted-parameter movement below 0.02 formal
sigma. These are numerical training tolerances, not proof that the timing model is adequate.
The parameter-step helper excludes JUMPs; inspect their values and degeneracies separately.

In [ ]:
source = ROOT/'exports/05_reviewed/data'
from pleb.residual_export_worker import parameters, maximum_step
model_rows = []
for psr in PULSARS:
    work = lab.simulation.project(ROOT/'final_model_fits'/psr)
    par, tim = source/psr/f'{psr}.par', source/psr/f'{psr}_all.tim'
    history, previous, last_wrms = [], None, None
    chosen = par
    for iteration in range(1,9):
        label = f'pass{iteration:02}'
        residuals, stats = lab.simulation.fit_tempo2(work,chosen,tim,label,epoch='58730')
        fitted = work.results/label/'new.par'
        current = parameters(fitted,(work.results/label/'tempo2.log').read_text())
        step = maximum_step(previous,current) if previous is not None else None
        relative = abs(stats['wrms_us']-last_wrms)/max(abs(last_wrms),1e-9) if last_wrms is not None else None
        history.append(dict(iteration=iteration,wrms_us=stats['wrms_us'],
                            max_step_sigma=step,relative_wrms_change=relative,n=stats['n']))
        if iteration>=3 and relative<1e-5 and step<0.02:
            chosen = fitted
            break
        chosen,previous,last_wrms = fitted,current,stats['wrms_us']
    else:
        raise RuntimeError(f'{psr}: fit did not converge; inspect the logs, do not approve it')
    assert len({h['n'] for h in history})==1
    display(pd.DataFrame(history))
    lab.simulation.plot_residuals(residuals,psr+': final retained TOAs')
    model_rows.append(dict(pulsar=psr,**{k:stats[k] for k in ['n','k','chisq','red_chi2','wrms_us']}))
    fitdir = ROOT/'fitted_pars'/psr
    fitdir.mkdir(parents=True,exist_ok=True)
    shutil.copy2(chosen,fitdir/'new.par')
    (fitdir/'convergence.json').write_text(json.dumps(history,indent=2))
    # Reload the saved coefficients without fitting again. Agreement of an
    # in-memory fit alone would not validate the PAR being distributed.
    reloaded, reload_stats = lab.simulation.fit_tempo2(work,fitdir/'new.par',tim,
        'saved_model_reload',epoch='58730',do_fit=False)
    assert abs(reload_stats['wrms_us']-stats['wrms_us'])<0.005
    assert (fitdir/'new.par').exists()
    print((fitdir/'new.par').read_text())
display(pd.DataFrame(model_rows))
pd.DataFrame(model_rows).to_csv(ROOT/'final_fit_metrics.csv',index=False)

**Model review:** inspect the saved logs and fitted PARs, compare values to the starting
models and verify that the selected rows did not change. Set the checkpoint below only
when satisfied. For timing-model candidate searches, use the separate timing-course
notebook 04; compare candidates on identical TOAs and document any accepted new terms.
This exercise starts from an independently specified model rather than teaching
automatic acceptance of model complexity.

If a nonlinear fit oscillates or fails, stop and review it. A previously validated
saved model can be an explicit alternative only after checking the exact TIM inputs,
effective clocks and model provenance against its saved signature. Mark that model as
reused, not newly converged. Export agreement is separate from model adequacy.

In [ ]:
MODELS_REVIEWED = False
MODEL_REVIEWER = ''
MODEL_REASON = ''
assert MODELS_REVIEWED and MODEL_REVIEWER and MODEL_REASON, 'Review the models before committing them'
model_repo = ROOT/'model-worktree'
lab.command(['git','-C',REPO,'worktree','add','-b','release_model',model_repo,'reviewed'])
for psr in PULSARS:
    shutil.copy2(ROOT/'fitted_pars'/psr/'new.par',model_repo/'data'/psr/f'{psr}.par')
lab.command(['git','-C',model_repo,'add','data'])
lab.command(['git','-C',model_repo,'commit','-m','Record reviewed final fitted timing models'])
lab.command(['git','-C',REPO,'worktree','remove',model_repo])
assert not lab.command(['git','-C',REPO,'diff','reviewed','release_model','--','data/**/*.tim']).strip()
(ROOT/'model_approval.json').write_text(json.dumps(dict(training_only=True,
    reviewer=MODEL_REVIEWER,reason=MODEL_REASON,metrics=model_rows),indent=2))

**Your task:** explain why a detector rerun, a numerical timing fit and accepting a new
model term are three different decisions. Identify the PAR and TIM pair you would
supply to someone analysing this release.